In [52]:
import math
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

NOTEBOOK_PATH = Path("/Users/nick/Documents/www/alphakit/notebooks/binance_arbitrage_test.ipynb")
DATA_DIR = NOTEBOOK_PATH.parent / "data"
DATA_DIR.mkdir(exist_ok=True)

SPOT_BASE_URL = "https://api.binance.com"
FUTURES_BASE_URL = "https://fapi.binance.com"
KLINE_LIMIT = 1000
DEFAULT_KLINE_INTERVAL = "1m"
PAIR_CANDIDATE_COUNT: int | None = None
CAPITAL_SPLIT_SPOT = 0.5
CAPITAL_SPLIT_FUTURES = 0.5
MIN_WEEKLY_COVERAGE_RATIO = 0.98
BINANCE_MAX_RETRIES = 6
BINANCE_RETRY_SLEEP_SECONDS = 2.0

CALIBRATION_WEEK_START_ISO = "2026-03-23T00:00:00+00:00"
CALIBRATION_WEEK_END_ISO = "2026-03-29T23:59:00+00:00"
CALIBRATION_WEEK_END_EXCLUSIVE_ISO = "2026-03-30T00:00:00+00:00"
TRADING_WEEK_START_ISO = "2026-03-30T00:00:00+00:00"
TRADING_WEEK_END_ISO = "2026-04-05T23:59:00+00:00"
TRADING_WEEK_END_EXCLUSIVE_ISO = "2026-04-06T00:00:00+00:00"
DOWNLOAD_START_ISO = CALIBRATION_WEEK_START_ISO
DOWNLOAD_END_EXCLUSIVE_ISO = TRADING_WEEK_END_EXCLUSIVE_ISO

SPOT_TAKER_FEE = 0.001
SPOT_MAKER_FEE = 0.001
FUTURES_TAKER_FEE = 0.0005
FUTURES_MAKER_FEE = 0.0002
TAKER_ROUNDTRIP_BPS = 2.0 * (SPOT_TAKER_FEE + FUTURES_TAKER_FEE) * 10_000


@dataclass(frozen=True)
class PairCandidate:
    spot_symbol: str
    futures_symbol: str
    base_asset: str
    quote_asset: str
    spot_quote_volume: float
    futures_quote_volume: float


session = requests.Session()
session.headers.update({"User-Agent": "alphakit-arb-research/0.1"})


def interval_to_milliseconds(interval: str) -> int:
    unit = interval[-1]
    value = int(interval[:-1])
    multipliers = {
        "m": 60 * 1000,
        "h": 60 * 60 * 1000,
        "d": 24 * 60 * 60 * 1000,
        "w": 7 * 24 * 60 * 60 * 1000,
    }
    if unit not in multipliers:
        raise ValueError(f"Unsupported interval={interval}")
    return value * multipliers[unit]


def interval_to_pandas_freq(interval: str) -> str:
    unit = interval[-1]
    value = interval[:-1]
    freq_map = {
        "m": "min",
        "h": "h",
        "d": "d",
        "w": "w",
    }
    if unit not in freq_map:
        raise ValueError(f"Unsupported interval={interval}")
    return f"{value}{freq_map[unit]}"


def iso_to_timestamp_ms(value: str) -> int:
    return int(pd.Timestamp(value).timestamp() * 1000)


def iso_to_timestamp(value: str) -> pd.Timestamp:
    return pd.Timestamp(value)


def expected_rows_for_range(start_iso: str, end_exclusive_iso: str, interval: str) -> int:
    start = iso_to_timestamp(start_iso)
    end = iso_to_timestamp(end_exclusive_iso)
    step = interval_to_milliseconds(interval)
    return int((end - start) // pd.Timedelta(milliseconds=step))


def binance_get(base_url: str, path: str, params: dict[str, Any] | None = None) -> Any:
    last_error: Exception | None = None
    for attempt in range(BINANCE_MAX_RETRIES):
        try:
            response = session.get(f"{base_url}{path}", params=params, timeout=30)
            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")
                sleep_seconds = float(retry_after) if retry_after else BINANCE_RETRY_SLEEP_SECONDS * (attempt + 1)
                time.sleep(sleep_seconds)
                continue
            response.raise_for_status()
            return response.json()
        except requests.HTTPError as exc:
            last_error = exc
            if exc.response is not None and exc.response.status_code in {418, 429}:
                retry_after = exc.response.headers.get("Retry-After")
                sleep_seconds = float(retry_after) if retry_after else BINANCE_RETRY_SLEEP_SECONDS * (attempt + 1)
                time.sleep(sleep_seconds)
                continue
            raise
        except requests.RequestException as exc:
            last_error = exc
            time.sleep(BINANCE_RETRY_SLEEP_SECONDS * (attempt + 1))
    if last_error is not None:
        raise last_error
    raise RuntimeError("Binance request failed without an exception")


def get_spot_exchange_info() -> pd.DataFrame:
    payload = binance_get(SPOT_BASE_URL, "/api/v3/exchangeInfo")
    rows: list[dict[str, Any]] = []
    for symbol in payload["symbols"]:
        if symbol["status"] != "TRADING":
            continue
        rows.append(
            {
                "symbol": symbol["symbol"],
                "base_asset": symbol["baseAsset"],
                "quote_asset": symbol["quoteAsset"],
                "market_type": "spot",
                "is_spot_trading_allowed": symbol.get("isSpotTradingAllowed", False),
            }
        )
    return pd.DataFrame(rows)


def get_futures_exchange_info() -> pd.DataFrame:
    payload = binance_get(FUTURES_BASE_URL, "/fapi/v1/exchangeInfo")
    rows: list[dict[str, Any]] = []
    for symbol in payload["symbols"]:
        if symbol["status"] != "TRADING" or symbol.get("contractType") != "PERPETUAL":
            continue
        rows.append(
            {
                "symbol": symbol["symbol"],
                "base_asset": symbol["baseAsset"],
                "quote_asset": symbol["quoteAsset"],
                "market_type": "usd_m_perpetual",
                "contract_type": symbol.get("contractType"),
            }
        )
    return pd.DataFrame(rows)


def get_spot_ticker_24h() -> pd.DataFrame:
    ticker = pd.DataFrame(binance_get(SPOT_BASE_URL, "/api/v3/ticker/24hr"))
    numeric_cols = ["lastPrice", "quoteVolume", "volume", "count"]
    for col in numeric_cols:
        ticker[col] = pd.to_numeric(ticker[col], errors="coerce")
    return ticker[["symbol", "lastPrice", "quoteVolume", "volume", "count"]].rename(
        columns={
            "lastPrice": "spot_last_price",
            "quoteVolume": "spot_quote_volume",
            "volume": "spot_base_volume",
            "count": "spot_trade_count",
        }
    )


def get_futures_ticker_24h() -> pd.DataFrame:
    ticker = pd.DataFrame(binance_get(FUTURES_BASE_URL, "/fapi/v1/ticker/24hr"))
    numeric_cols = ["lastPrice", "quoteVolume", "volume", "count"]
    for col in numeric_cols:
        ticker[col] = pd.to_numeric(ticker[col], errors="coerce")
    return ticker[["symbol", "lastPrice", "quoteVolume", "volume", "count"]].rename(
        columns={
            "lastPrice": "futures_last_price",
            "quoteVolume": "futures_quote_volume",
            "volume": "futures_base_volume",
            "count": "futures_trade_count",
        }
    )


def get_symbol_metadata() -> dict[str, pd.DataFrame]:
    spot_info = get_spot_exchange_info()
    futures_info = get_futures_exchange_info()
    spot_ticker = get_spot_ticker_24h()
    futures_ticker = get_futures_ticker_24h()

    spot_markets = spot_info.merge(spot_ticker, on="symbol", how="left")
    futures_markets = futures_info.merge(futures_ticker, on="symbol", how="left")
    overlap = spot_markets.merge(
        futures_markets,
        on=["symbol", "base_asset", "quote_asset"],
        suffixes=("_spot", "_futures"),
        how="inner",
    )
    overlap = overlap.sort_values(
        ["spot_quote_volume", "futures_quote_volume"], ascending=False
    ).reset_index(drop=True)

    return {
        "spot": spot_markets,
        "futures": futures_markets,
        "spot_perp_overlap": overlap,
    }


market_data = get_symbol_metadata()
spot_markets = market_data["spot"]
futures_markets = market_data["futures"]
spot_perp_overlap = market_data["spot_perp_overlap"]
effective_pair_count = len(spot_perp_overlap) if PAIR_CANDIDATE_COUNT is None else min(PAIR_CANDIDATE_COUNT, len(spot_perp_overlap))
expected_download_rows = expected_rows_for_range(
    DOWNLOAD_START_ISO,
    DOWNLOAD_END_EXCLUSIVE_ISO,
    DEFAULT_KLINE_INTERVAL,
)

print(f"Spot markets: {len(spot_markets):,}")
print(f"USD-M perpetual markets: {len(futures_markets):,}")
print(f"Spot/perpetual overlaps: {len(spot_perp_overlap):,}")
print(f"Default candle interval: {DEFAULT_KLINE_INTERVAL}")
print(f"Pair candidates to evaluate: {effective_pair_count:,}")
print(f"Calibration week: {CALIBRATION_WEEK_START_ISO} -> {CALIBRATION_WEEK_END_ISO}")
print(f"Trading week: {TRADING_WEEK_START_ISO} -> {TRADING_WEEK_END_ISO}")
print(f"Expected cached rows per market: {expected_download_rows:,}")
print(f"Capital split: {CAPITAL_SPLIT_SPOT:.0%} spot / {CAPITAL_SPLIT_FUTURES:.0%} futures")
print(f"Execution model: taker / taker")
print(f"Parquet cache directory: {DATA_DIR}")

spot_perp_overlap[
    [
        "symbol",
        "base_asset",
        "quote_asset",
        "spot_last_price",
        "futures_last_price",
        "spot_quote_volume",
        "futures_quote_volume",
    ]
] .head(20)

Spot markets: 1,420
USD-M perpetual markets: 577
Spot/perpetual overlaps: 406
Default candle interval: 1m
Pair candidates to evaluate: 406
Calibration week: 2026-03-23T00:00:00+00:00 -> 2026-03-29T23:59:00+00:00
Trading week: 2026-03-30T00:00:00+00:00 -> 2026-04-05T23:59:00+00:00
Expected cached rows per market: 20,160
Capital split: 50% spot / 50% futures
Execution model: taker / taker
Parquet cache directory: /Users/nick/Documents/www/alphakit/notebooks/data


,symbol,base_asset,quote_asset,spot_last_price,futures_last_price,spot_quote_volume,futures_quote_volume
0,BTCUSDT,BTC,USDT,70899.08000,70866.400000,1.005588e+09,9.877591e+09
1,ETHUSDT,ETH,USDT,2186.94000,2185.710000,7.461689e+08,1.001089e+10
2,ETHUSDC,ETH,USDC,2187.86000,2186.550000,4.318912e+08,2.887330e+09
3,USDCUSDT,USDC,USDT,0.99950,0.999042,3.385484e+08,2.516521e+06
4,BTCUSDC,BTC,USDC,70937.16000,70896.700000,2.045718e+08,2.032450e+09
5,SOLUSDT,SOL,USDT,81.68000,81.640000,1.958836e+08,1.644806e+09
6,XRPUSDT,XRP,USDT,1.32590,1.325000,1.097512e+08,5.256611e+08
7,DOGEUSDT,DOGE,USDT,0.09071,0.090650,7.974999e+07,5.027595e+08
8,ZECUSDT,ZEC,USDT,362.54000,362.360000,7.512468e+07,6.888557e+08
9,BNBUSDT,BNB,USDT,592.13000,592.280000,6.627023e+07,2.832382e+08


In [47]:
pair_candidates = spot_perp_overlap.head(effective_pair_count).copy()

pair_candidates["implied_two_market_edge_bps"] = (
    (pair_candidates["futures_last_price"] - pair_candidates["spot_last_price"])
    / pair_candidates["spot_last_price"]
    * 10_000
)

print(f"Two-market candidates selected: {len(pair_candidates):,}")
display(
    pair_candidates[
        [
            "symbol",
            "base_asset",
            "quote_asset",
            "spot_last_price",
            "futures_last_price",
            "implied_two_market_edge_bps",
            "spot_quote_volume",
            "futures_quote_volume",
        ]
    ]
)

Two-market candidates selected: 406


,symbol,base_asset,quote_asset,spot_last_price,futures_last_price,implied_two_market_edge_bps,spot_quote_volume,futures_quote_volume
0,BTCUSDT,BTC,USDT,70887.420000,70848.700000,-5.462182,1.020250e+09,9.782805e+09
1,ETHUSDT,ETH,USDT,2184.430000,2183.140000,-5.905431,7.423652e+08,9.906577e+09
2,ETHUSDC,ETH,USDC,2185.430000,2184.220000,-5.536668,4.291065e+08,2.867285e+09
3,USDCUSDT,USDC,USDT,0.999500,0.999041,-4.592296,3.398439e+08,2.472385e+06
4,BTCUSDC,BTC,USDC,70918.200000,70887.500000,-4.328931,2.041385e+08,2.030239e+09
...,...,...,...,...,...,...,...,...
401,PNUTUSDC,PNUT,USDC,0.040300,0.040220,-19.851117,5.211071e+04,6.473714e+05
402,ATAUSDT,ATA,USDT,0.008600,0.008680,93.023256,5.126717e+04,7.748390e+05
403,ARKUSDT,ARK,USDT,0.163300,0.162900,-24.494795,3.450908e+04,4.231980e+05
404,BOMEUSDC,BOME,USDC,0.000378,0.000378,0.000000,2.564046e+04,5.530476e+05


In [53]:
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_PARALLEL_DOWNLOADS = 5


def cache_file_path(
    symbol: str,
    market_type: str,
    interval: str,
    start_iso: str,
    end_exclusive_iso: str,
) -> Path:
    safe_start = start_iso.replace(":", "-")
    safe_end = end_exclusive_iso.replace(":", "-")
    return DATA_DIR / f"{symbol}__{market_type}__{interval}__{safe_start}__{safe_end}.parquet"


def validate_cached_klines(
    frame: pd.DataFrame,
    symbol: str,
    market_type: str,
    interval: str,
    start_iso: str,
    end_exclusive_iso: str,
) -> bool:
    if frame.empty:
        return False
    required_columns = {
        "timestamp",
        "symbol",
        "market_type",
        "interval",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "quote_asset_volume",
        "trade_count",
    }
    if not required_columns.issubset(frame.columns):
        return False

    working = frame.copy()
    working["timestamp"] = pd.to_datetime(working["timestamp"], utc=True)
    working = working.sort_values("timestamp").drop_duplicates(subset=["timestamp"])

    if working["symbol"].nunique() != 1 or working["symbol"].iloc[0] != symbol:
        return False
    if working["market_type"].nunique() != 1 or working["market_type"].iloc[0] != market_type:
        return False
    if working["interval"].nunique() != 1 or working["interval"].iloc[0] != interval:
        return False

    start_ts = iso_to_timestamp(start_iso)
    end_exclusive_ts = iso_to_timestamp(end_exclusive_iso)
    if working["timestamp"].iloc[0] < start_ts:
        return False
    if working["timestamp"].iloc[-1] >= end_exclusive_ts:
        return False
    if working["timestamp"].is_monotonic_increasing is False:
        return False

    delta = working["timestamp"].diff().dropna()
    interval_delta = pd.Timedelta(milliseconds=interval_to_milliseconds(interval))
    if not delta.empty and (delta < interval_delta).any():
        return False
    return True


def fetch_klines(
    symbol: str,
    market_type: str,
    interval: str = DEFAULT_KLINE_INTERVAL,
    start_iso: str = DOWNLOAD_START_ISO,
    end_exclusive_iso: str = DOWNLOAD_END_EXCLUSIVE_ISO,
) -> pd.DataFrame:
    if market_type == "spot":
        base_url = SPOT_BASE_URL
        path = "/api/v3/klines"
    elif market_type == "usd_m_perpetual":
        base_url = FUTURES_BASE_URL
        path = "/fapi/v1/klines"
    else:
        raise ValueError(f"Unsupported market_type={market_type}")

    interval_ms = interval_to_milliseconds(interval)
    request_limit = min(KLINE_LIMIT, 1000)
    start_time_ms = iso_to_timestamp_ms(start_iso)
    end_time_ms = iso_to_timestamp_ms(end_exclusive_iso)
    frames: list[pd.DataFrame] = []
    cursor = start_time_ms

    while cursor < end_time_ms:
        payload = binance_get(
            base_url,
            path,
            {
                "symbol": symbol,
                "interval": interval,
                "startTime": cursor,
                "endTime": end_time_ms,
                "limit": request_limit,
            },
        )
        if not payload:
            break

        frame = pd.DataFrame(
            payload,
            columns=[
                "open_time",
                "open",
                "high",
                "low",
                "close",
                "volume",
                "close_time",
                "quote_asset_volume",
                "trade_count",
                "taker_buy_base_volume",
                "taker_buy_quote_volume",
                "ignore",
            ],
        )
        frames.append(frame)
        last_open_time = int(frame["open_time"].iloc[-1])
        cursor = last_open_time + interval_ms
        if len(frame) < request_limit:
            break

    if not frames:
        return pd.DataFrame()

    klines = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["open_time"])
    numeric_cols = [
        "open",
        "high",
        "low",
        "close",
        "volume",
        "quote_asset_volume",
        "trade_count",
        "taker_buy_base_volume",
        "taker_buy_quote_volume",
    ]
    for col in numeric_cols:
        klines[col] = pd.to_numeric(klines[col], errors="coerce")
    klines["timestamp"] = pd.to_datetime(klines["open_time"], unit="ms", utc=True)
    klines = klines[
        (klines["timestamp"] >= iso_to_timestamp(start_iso))
        & (klines["timestamp"] < iso_to_timestamp(end_exclusive_iso))
    ]
    klines["symbol"] = symbol
    klines["market_type"] = market_type
    klines["interval"] = interval
    return klines[
        [
            "timestamp",
            "symbol",
            "market_type",
            "interval",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "quote_asset_volume",
            "trade_count",
        ]
    ].sort_values("timestamp").reset_index(drop=True)


def load_or_download_klines(
    symbol: str,
    market_type: str,
    interval: str = DEFAULT_KLINE_INTERVAL,
    start_iso: str = DOWNLOAD_START_ISO,
    end_exclusive_iso: str = DOWNLOAD_END_EXCLUSIVE_ISO,
) -> tuple[pd.DataFrame, str]:
    cache_path = cache_file_path(symbol, market_type, interval, start_iso, end_exclusive_iso)
    if cache_path.exists():
        cached = pd.read_parquet(cache_path)
        if validate_cached_klines(cached, symbol, market_type, interval, start_iso, end_exclusive_iso):
            cached["timestamp"] = pd.to_datetime(cached["timestamp"], utc=True)
            return cached.sort_values("timestamp").reset_index(drop=True), "cache"

    downloaded = fetch_klines(
        symbol=symbol,
        market_type=market_type,
        interval=interval,
        start_iso=start_iso,
        end_exclusive_iso=end_exclusive_iso,
    )
    if not validate_cached_klines(downloaded, symbol, market_type, interval, start_iso, end_exclusive_iso):
        raise ValueError(f"Downloaded candles failed validation for {symbol} {market_type}")
    downloaded.to_parquet(cache_path, index=False)
    return downloaded, "downloaded"


def collect_market_download_plan(pair_candidates: pd.DataFrame) -> pd.DataFrame:
    download_rows: list[dict[str, str]] = []
    for row in pair_candidates.itertuples(index=False):
        download_rows.append({"symbol": row.symbol, "market_type": "spot", "watch_type": "pair"})
        download_rows.append(
            {
                "symbol": row.symbol,
                "market_type": "usd_m_perpetual",
                "watch_type": "pair",
            }
        )
    plan = pd.DataFrame(download_rows).drop_duplicates().reset_index(drop=True)
    return plan.sort_values(["market_type", "symbol"]).reset_index(drop=True)


def load_plan_row(row: Any) -> tuple[tuple[str, str], pd.DataFrame, str]:
    key = (row.symbol, row.market_type)
    frame, source = load_or_download_klines(row.symbol, row.market_type)
    return key, frame, source


market_download_plan = collect_market_download_plan(pair_candidates)
print(f"Markets to materialize: {len(market_download_plan):,}")
print(f"Requested candle interval: {DEFAULT_KLINE_INTERVAL}")
print(f"Cache directory: {DATA_DIR}")
print(f"Parallel downloads: {MAX_PARALLEL_DOWNLOADS}")
display(market_download_plan)

market_candles: dict[tuple[str, str], pd.DataFrame] = {}
cache_stats = {"cache": 0, "downloaded": 0}
errors: list[str] = []

with ThreadPoolExecutor(max_workers=MAX_PARALLEL_DOWNLOADS) as executor:
    futures = {
        executor.submit(load_plan_row, row): (row.symbol, row.market_type)
        for row in market_download_plan.itertuples(index=False)
    }
    completed = 0
    total = len(futures)
    for future in as_completed(futures):
        symbol, market_type = futures[future]
        completed += 1
        try:
            key, frame, source = future.result()
            market_candles[key] = frame
            cache_stats[source] += 1
        except Exception as exc:
            errors.append(f"{symbol} {market_type}: {exc}")
        if completed % 50 == 0 or completed == total:
            print(f"Completed {completed:,}/{total:,} market loads")

if errors:
    raise RuntimeError("Failed market loads:\n" + "\n".join(errors[:20]))

print(f"Loaded from cache: {cache_stats['cache']:,}")
print(f"Downloaded now: {cache_stats['downloaded']:,}")

candle_coverage = pd.DataFrame(
    [
        {
            "symbol": symbol,
            "market_type": market_type,
            "interval": frame["interval"].iloc[0] if not frame.empty else None,
            "rows": len(frame),
            "start": frame["timestamp"].min() if not frame.empty else pd.NaT,
            "end": frame["timestamp"].max() if not frame.empty else pd.NaT,
        }
        for (symbol, market_type), frame in market_candles.items()
    ]
).sort_values(["market_type", "symbol"])

display(candle_coverage.head(20))

Markets to materialize: 812
Requested candle interval: 1m
Cache directory: /Users/nick/Documents/www/alphakit/notebooks/data
Parallel downloads: 5


,symbol,market_type,watch_type
0,0GUSDT,spot,pair
1,1000CATUSDT,spot,pair
2,1000CHEEMSUSDT,spot,pair
3,1000SATSUSDT,spot,pair
4,1INCHUSDT,spot,pair
...,...,...,...
807,ZKPUSDT,usd_m_perpetual,pair
808,ZKUSDT,usd_m_perpetual,pair
809,ZROUSDT,usd_m_perpetual,pair
810,ZRXUSDT,usd_m_perpetual,pair


Completed 50/812 market loads
Completed 100/812 market loads
Completed 150/812 market loads
Completed 200/812 market loads
Completed 250/812 market loads
Completed 300/812 market loads
Completed 350/812 market loads
Completed 400/812 market loads
Completed 450/812 market loads
Completed 500/812 market loads
Completed 550/812 market loads
Completed 600/812 market loads
Completed 650/812 market loads
Completed 700/812 market loads
Completed 750/812 market loads
Completed 800/812 market loads
Completed 812/812 market loads
Loaded from cache: 527
Downloaded now: 285


,symbol,market_type,interval,rows,start,end
3,0GUSDT,spot,1m,20160,2026-03-23 00:00:00+00:00,2026-04-05 23:59:00+00:00
1,1000CATUSDT,spot,1m,20160,2026-03-23 00:00:00+00:00,2026-04-05 23:59:00+00:00
2,1000CHEEMSUSDT,spot,1m,20160,2026-03-23 00:00:00+00:00,2026-04-05 23:59:00+00:00
0,1000SATSUSDT,spot,1m,20160,2026-03-23 00:00:00+00:00,2026-04-05 23:59:00+00:00
4,1INCHUSDT,spot,1m,20160,2026-03-23 00:00:00+00:00,2026-04-05 23:59:00+00:00
7,1MBABYDOGEUSDT,spot,1m,20160,2026-03-23 00:00:00+00:00,2026-04-05 23:59:00+00:00
9,2ZUSDT,spot,1m,20160,2026-03-23 00:00:00+00:00,2026-04-05 23:59:00+00:00
5,AAVEUSDC,spot,1m,20160,2026-03-23 00:00:00+00:00,2026-04-05 23:59:00+00:00
6,AAVEUSDT,spot,1m,20160,2026-03-23 00:00:00+00:00,2026-04-05 23:59:00+00:00
8,ACEUSDT,spot,1m,20160,2026-03-23 00:00:00+00:00,2026-04-05 23:59:00+00:00


In [54]:
def build_pair_frame(
    pair_row: pd.Series,
    market_candles: dict[tuple[str, str], pd.DataFrame],
) -> pd.DataFrame:
    spot = market_candles.get((pair_row["symbol"], "spot"), pd.DataFrame()).copy()
    perp = market_candles.get((pair_row["symbol"], "usd_m_perpetual"), pd.DataFrame()).copy()
    if spot.empty or perp.empty:
        return pd.DataFrame()

    merged = spot[["timestamp", "close"]].rename(columns={"close": "spot_close"}).merge(
        perp[["timestamp", "close"]].rename(columns={"close": "perp_close"}),
        on="timestamp",
        how="inner",
    )
    if merged.empty:
        return pd.DataFrame()

    merged["symbol"] = pair_row["symbol"]
    merged["base_asset"] = pair_row["base_asset"]
    merged["spread_bps"] = (merged["perp_close"] - merged["spot_close"]) / merged["spot_close"] * 10_000
    merged["week_bucket"] = np.where(
        merged["timestamp"] < iso_to_timestamp(CALIBRATION_WEEK_END_EXCLUSIVE_ISO),
        "calibration",
        "trading",
    )
    calibration_expected = expected_rows_for_range(
        CALIBRATION_WEEK_START_ISO,
        CALIBRATION_WEEK_END_EXCLUSIVE_ISO,
        DEFAULT_KLINE_INTERVAL,
    )
    trading_expected = expected_rows_for_range(
        TRADING_WEEK_START_ISO,
        TRADING_WEEK_END_EXCLUSIVE_ISO,
        DEFAULT_KLINE_INTERVAL,
    )
    calibration = merged[merged["week_bucket"] == "calibration"].copy()
    trading = merged[merged["week_bucket"] == "trading"].copy()
    if len(calibration) < calibration_expected * MIN_WEEKLY_COVERAGE_RATIO:
        return pd.DataFrame()
    if len(trading) < trading_expected * MIN_WEEKLY_COVERAGE_RATIO:
        return pd.DataFrame()
    return merged.sort_values("timestamp").reset_index(drop=True)


def simulate_threshold_strategy(
    frame: pd.DataFrame,
    entry_threshold_bps: float,
    target_spread_bps: float,
    starting_capital: float = 1_000.0,
) -> tuple[dict[str, Any], pd.DataFrame]:
    balance = starting_capital
    entry_idx: int | None = None
    trades: list[dict[str, Any]] = []
    working = frame.sort_values("timestamp").reset_index(drop=True).copy()

    for idx, row in working.iterrows():
        spread_bps = float(row["spread_bps"])
        if entry_idx is None:
            if spread_bps >= entry_threshold_bps:
                entry_idx = idx
            continue

        should_exit = False
        if spread_bps <= target_spread_bps:
            should_exit = True
        elif idx == len(working) - 1:
            should_exit = True

        if not should_exit or entry_idx is None:
            continue

        entry = working.iloc[entry_idx]
        exit_row = row

        spot_capital = balance * CAPITAL_SPLIT_SPOT
        futures_capital = balance * CAPITAL_SPLIT_FUTURES
        spot_entry_price = float(entry["spot_close"])
        spot_exit_price = float(exit_row["spot_close"])
        perp_entry_price = float(entry["perp_close"])
        perp_exit_price = float(exit_row["perp_close"])

        spot_quantity = 0.0 if spot_entry_price == 0 else spot_capital / spot_entry_price
        perp_quantity = 0.0 if perp_entry_price == 0 else futures_capital / perp_entry_price

        spot_entry_fee = spot_capital * SPOT_TAKER_FEE
        spot_exit_notional = spot_quantity * spot_exit_price
        spot_exit_fee = spot_exit_notional * SPOT_TAKER_FEE
        perp_entry_fee = futures_capital * FUTURES_TAKER_FEE
        perp_exit_notional = perp_quantity * perp_exit_price
        perp_exit_fee = perp_exit_notional * FUTURES_TAKER_FEE

        spot_gross_pnl_usd = spot_quantity * (spot_exit_price - spot_entry_price)
        perp_gross_pnl_usd = perp_quantity * (perp_entry_price - perp_exit_price)
        gross_pnl_usd = spot_gross_pnl_usd + perp_gross_pnl_usd
        total_fees_usd = spot_entry_fee + spot_exit_fee + perp_entry_fee + perp_exit_fee
        net_pnl_usd = gross_pnl_usd - total_fees_usd
        balance += net_pnl_usd

        gross_pnl_bps = gross_pnl_usd / starting_capital * 10_000
        net_pnl_bps = net_pnl_usd / starting_capital * 10_000
        holding_timedelta = exit_row["timestamp"] - entry["timestamp"]
        trades.append(
            {
                "symbol": entry["symbol"],
                "entry_time": entry["timestamp"],
                "exit_time": exit_row["timestamp"],
                "entry_spread_bps": float(entry["spread_bps"]),
                "exit_spread_bps": float(exit_row["spread_bps"]),
                "target_spread_bps": target_spread_bps,
                "entry_threshold_bps": entry_threshold_bps,
                "spot_capital_usd": spot_capital,
                "futures_capital_usd": futures_capital,
                "spot_quantity": spot_quantity,
                "perp_quantity": perp_quantity,
                "spot_gross_pnl_usd": spot_gross_pnl_usd,
                "perp_gross_pnl_usd": perp_gross_pnl_usd,
                "gross_pnl_usd": gross_pnl_usd,
                "fees_usd": total_fees_usd,
                "net_pnl_usd": net_pnl_usd,
                "gross_pnl_bps": gross_pnl_bps,
                "net_pnl_bps": net_pnl_bps,
                "ending_balance": balance,
                "holding_bars": idx - entry_idx,
                "holding_timedelta": holding_timedelta,
                "holding_minutes": holding_timedelta.total_seconds() / 60,
                "winner": net_pnl_usd > 0,
            }
        )
        entry_idx = None

    trades_df = pd.DataFrame(trades)
    trade_count = len(trades_df)
    summary = {
        "trade_count": trade_count,
        "ending_balance_usd": balance,
        "net_pnl_usd": balance - starting_capital,
        "return_pct": (balance / starting_capital - 1.0) * 100,
        "winner_count": int(trades_df["winner"].sum()) if trade_count else 0,
        "loser_count": trade_count - int(trades_df["winner"].sum()) if trade_count else 0,
        "win_rate_pct": float(trades_df["winner"].mean() * 100) if trade_count else 0.0,
        "avg_trade_pnl_usd": float(trades_df["net_pnl_usd"].mean()) if trade_count else 0.0,
        "avg_holding_minutes": float(trades_df["holding_minutes"].mean()) if trade_count else 0.0,
        "median_holding_minutes": float(trades_df["holding_minutes"].median()) if trade_count else 0.0,
        "gross_pnl_usd_sum": float(trades_df["gross_pnl_usd"].sum()) if trade_count else 0.0,
        "net_pnl_usd_sum": float(trades_df["net_pnl_usd"].sum()) if trade_count else 0.0,
    }
    return summary, trades_df


def build_threshold_grid(
    calibration_frame: pd.DataFrame,
    target_spread_bps: float,
    fee_roundtrip_bps: float,
    grid_size: int = 25,
) -> np.ndarray:
    profitable = calibration_frame.loc[
        calibration_frame["spread_bps"] >= target_spread_bps + fee_roundtrip_bps,
        "spread_bps",
    ]
    if profitable.empty:
        return np.array([], dtype=float)

    quantiles = np.linspace(0.0, 1.0, grid_size)
    threshold_grid = np.quantile(profitable.to_numpy(), quantiles)
    threshold_grid = np.unique(np.round(threshold_grid, 6))
    threshold_grid = threshold_grid[threshold_grid >= target_spread_bps + fee_roundtrip_bps]
    return threshold_grid


def optimize_thresholds(
    pair_candidates: pd.DataFrame,
    market_candles: dict[tuple[str, str], pd.DataFrame],
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame], dict[str, pd.DataFrame]]:
    summary_rows: list[dict[str, Any]] = []
    pair_series: dict[str, pd.DataFrame] = {}
    threshold_search_logs: dict[str, pd.DataFrame] = {}

    for row in pair_candidates.itertuples(index=False):
        pair_row = pd.Series({
            "symbol": row.symbol,
            "base_asset": row.base_asset,
        })
        merged = build_pair_frame(pair_row, market_candles)
        if merged.empty:
            continue

        calibration = merged[merged["week_bucket"] == "calibration"].copy()
        trading = merged[merged["week_bucket"] == "trading"].copy()
        if calibration.empty or trading.empty:
            continue

        target_spread_bps = float(calibration["spread_bps"].mean())
        calibration_std = float(calibration["spread_bps"].std(ddof=0))
        threshold_grid = build_threshold_grid(calibration, target_spread_bps, TAKER_ROUNDTRIP_BPS)
        if threshold_grid.size == 0:
            merged["week_target_spread_bps"] = target_spread_bps
            pair_series[row.symbol] = merged
            summary_rows.append(
                {
                    "symbol": row.symbol,
                    "base_asset": row.base_asset,
                    "target_spread_bps": target_spread_bps,
                    "optimal_entry_threshold_bps": np.nan,
                    "calibration_trade_count": 0,
                    "calibration_return_pct": 0.0,
                    "calibration_net_pnl_usd": 0.0,
                    "calibration_max_spread_bps": float(calibration["spread_bps"].max()),
                    "calibration_mean_spread_bps": target_spread_bps,
                    "calibration_std_spread_bps": calibration_std,
                    "trading_trade_count": 0,
                    "trading_return_pct": 0.0,
                    "trading_net_pnl_usd": 0.0,
                    "trading_win_rate_pct": 0.0,
                    "trading_avg_holding_minutes": 0.0,
                }
            )
            threshold_search_logs[row.symbol] = pd.DataFrame()
            continue

        optimization_rows: list[dict[str, Any]] = []
        best_threshold = float(threshold_grid[0])
        best_score = -float("inf")
        best_calibration_summary: dict[str, Any] | None = None

        for threshold_bps in threshold_grid:
            calibration_summary, _ = simulate_threshold_strategy(
                calibration,
                entry_threshold_bps=float(threshold_bps),
                target_spread_bps=target_spread_bps,
            )
            optimization_rows.append(
                {
                    "entry_threshold_bps": float(threshold_bps),
                    "trade_count": calibration_summary["trade_count"],
                    "net_pnl_usd": calibration_summary["net_pnl_usd"],
                    "return_pct": calibration_summary["return_pct"],
                    "win_rate_pct": calibration_summary["win_rate_pct"],
                    "avg_holding_minutes": calibration_summary["avg_holding_minutes"],
                }
            )
            score = calibration_summary["net_pnl_usd"]
            if calibration_summary["trade_count"] == 0:
                score = -float("inf")
            if score > best_score:
                best_score = score
                best_threshold = float(threshold_bps)
                best_calibration_summary = calibration_summary

        merged["week_target_spread_bps"] = target_spread_bps
        merged["optimal_entry_threshold_bps"] = best_threshold
        merged["signal"] = np.where(
            (merged["week_bucket"] == "trading") & (merged["spread_bps"] >= best_threshold),
            -1,
            0,
        )
        pair_series[row.symbol] = merged
        threshold_search_logs[row.symbol] = pd.DataFrame(optimization_rows).sort_values(
            ["net_pnl_usd", "trade_count"],
            ascending=[False, False],
        )

        trading_summary, _ = simulate_threshold_strategy(
            trading,
            entry_threshold_bps=best_threshold,
            target_spread_bps=target_spread_bps,
        )
        summary_rows.append(
            {
                "symbol": row.symbol,
                "base_asset": row.base_asset,
                "target_spread_bps": target_spread_bps,
                "optimal_entry_threshold_bps": best_threshold,
                "calibration_trade_count": best_calibration_summary["trade_count"] if best_calibration_summary else 0,
                "calibration_return_pct": best_calibration_summary["return_pct"] if best_calibration_summary else 0.0,
                "calibration_net_pnl_usd": best_calibration_summary["net_pnl_usd"] if best_calibration_summary else 0.0,
                "calibration_max_spread_bps": float(calibration["spread_bps"].max()),
                "calibration_mean_spread_bps": target_spread_bps,
                "calibration_std_spread_bps": calibration_std,
                "trading_trade_count": trading_summary["trade_count"],
                "trading_return_pct": trading_summary["return_pct"],
                "trading_net_pnl_usd": trading_summary["net_pnl_usd"],
                "trading_win_rate_pct": trading_summary["win_rate_pct"],
                "trading_avg_holding_minutes": trading_summary["avg_holding_minutes"],
            }
        )

    if not summary_rows:
        empty_summary = pd.DataFrame(
            columns=[
                "symbol",
                "base_asset",
                "target_spread_bps",
                "optimal_entry_threshold_bps",
                "calibration_trade_count",
                "calibration_return_pct",
                "calibration_net_pnl_usd",
                "calibration_max_spread_bps",
                "calibration_mean_spread_bps",
                "calibration_std_spread_bps",
                "trading_trade_count",
                "trading_return_pct",
                "trading_net_pnl_usd",
                "trading_win_rate_pct",
                "trading_avg_holding_minutes",
            ]
        )
        return empty_summary, pair_series, threshold_search_logs

    summary = pd.DataFrame(summary_rows).sort_values(
        ["trading_net_pnl_usd", "calibration_net_pnl_usd"],
        ascending=[False, False],
    )
    return summary, pair_series, threshold_search_logs


pair_summary, pair_series, threshold_search_logs = optimize_thresholds(
    pair_candidates,
    market_candles,
)

print("Pair threshold optimization summary")
display(pair_summary)

top_optimized = pair_summary.head(5)
if not top_optimized.empty:
    fig = make_subplots(
        rows=len(top_optimized),
        cols=1,
        shared_xaxes=True,
        subplot_titles=list(top_optimized["symbol"]),
    )
    for idx, symbol in enumerate(top_optimized["symbol"], start=1):
        frame = pair_series[symbol]
        trading = frame[frame["week_bucket"] == "trading"]
        target_spread_bps = float(frame["week_target_spread_bps"].iloc[0])
        entry_threshold_bps = float(frame["optimal_entry_threshold_bps"].iloc[0])
        fig.add_trace(
            go.Scatter(
                x=trading["timestamp"],
                y=trading["spread_bps"],
                mode="lines",
                name=f"{symbol} spread",
                showlegend=False,
            ),
            row=idx,
            col=1,
        )
        fig.add_hline(y=target_spread_bps, line_width=1, line_dash="dot", line_color="gray", row=idx, col=1)
        fig.add_hline(y=entry_threshold_bps, line_width=1, line_dash="dash", line_color="crimson", row=idx, col=1)
    fig.update_layout(
        height=260 * len(top_optimized),
        width=1100,
        template="plotly_white",
        title="Trading-week spread in bps with calibrated target and entry threshold",
    )
    fig.show()

top_threshold_tables = top_optimized.head(3)["symbol"].tolist() if not top_optimized.empty else []
if top_threshold_tables:
    display({symbol: threshold_search_logs[symbol].head(10) for symbol in top_threshold_tables})

Pair threshold optimization summary


,symbol,base_asset,target_spread_bps,optimal_entry_threshold_bps,calibration_trade_count,calibration_return_pct,calibration_net_pnl_usd,calibration_max_spread_bps,calibration_mean_spread_bps,calibration_std_spread_bps,trading_trade_count,trading_return_pct,trading_net_pnl_usd,trading_win_rate_pct,trading_avg_holding_minutes
401,ATAUSDT,ATA,13.359356,99.009901,1320,22609.495391,226094.953907,240.963855,13.359356,81.339610,1296,44820.591947,448205.919466,100.000000,2.105710
215,ALICEUSDT,ALICE,-21.186034,8.857396,1246,5651.757328,56517.573283,97.431355,-21.186034,57.218418,1263,6607.289797,66072.897971,100.000000,3.977039
165,DYDXUSDT,DYDX,-13.416279,20.661157,1202,16655.343541,166553.435409,121.951220,-13.416279,52.204408,1090,5317.189109,53171.891090,100.000000,2.492661
302,1000SATSUSDT,1000SATS,-18.473382,16.835017,1173,4389.008169,43890.081689,112.359551,-18.473382,56.200982,1079,3563.888516,35638.885164,100.000000,4.490269
363,GTCUSDT,GTC,-7.083616,113.636364,294,2299.166615,22991.666146,256.410256,-7.083616,93.262950,306,3364.046625,33640.466245,99.673203,13.901961
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,FLOWUSDT,FLOW,0.087943,NaN,0,0.000000,0.000000,26.873694,0.087943,6.634387,0,0.000000,0.000000,0.000000,0.000000
239,APEUSDT,APE,-13.860421,NaN,0,0.000000,0.000000,11.848341,-13.860421,8.578525,0,0.000000,0.000000,0.000000,0.000000
340,NXPCUSDT,NXPC,-17.339664,NaN,0,0.000000,0.000000,7.246377,-17.339664,6.170735,0,0.000000,0.000000,0.000000,0.000000
380,ERAUSDT,ERA,-28.065883,NaN,0,0.000000,0.000000,0.000000,-28.065883,7.276745,0,0.000000,0.000000,0.000000,0.000000


{'ATAUSDT':    entry_threshold_bps  trade_count    net_pnl_usd    return_pct  win_rate_pct  avg_holding_minutes
 0            99.009901         1320  226094.953907  22609.495391         100.0             2.350000
 1           100.000000         1039   80190.609391   8019.060939         100.0             2.443696
 2           101.010101         1039   80190.609391   8019.060939         100.0             2.443696
 3           102.040816          820   35934.933963   3593.493396         100.0             2.557317
 4           105.263158          578   14100.071282   1410.007128         100.0             2.717993
 5           106.382979          479    9079.506558    907.950656         100.0             2.903967
 6           111.111111          472    8805.511474    880.551147         100.0             2.917373
 7           112.359551          239    2958.722033    295.872203         100.0             3.635983
 8           116.279070          146    1497.094273    149.709427         100.0 

In [55]:
STARTING_CAPITAL = 1_000.0
EXECUTION_STYLE = "taker"


def run_trading_week_backtest(
    pair_summary: pd.DataFrame,
    pair_series: dict[str, pd.DataFrame],
    starting_capital: float = STARTING_CAPITAL,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    trade_summaries: list[dict[str, Any]] = []
    trade_logs: dict[str, pd.DataFrame] = {}

    for row in pair_summary.itertuples(index=False):
        if pd.isna(row.optimal_entry_threshold_bps):
            trade_log = pd.DataFrame()
            trade_logs[row.symbol] = trade_log
            trade_summaries.append(
                {
                    "symbol": row.symbol,
                    "strategy_type": "pair",
                    "execution_style": EXECUTION_STYLE,
                    "optimal_entry_threshold_bps": np.nan,
                    "target_spread_bps": row.target_spread_bps,
                    "starting_balance_usd": starting_capital,
                    "spot_allocation_usd": starting_capital * CAPITAL_SPLIT_SPOT,
                    "futures_allocation_usd": starting_capital * CAPITAL_SPLIT_FUTURES,
                    "ending_balance_usd": starting_capital,
                    "net_pnl_usd": 0.0,
                    "return_pct": 0.0,
                    "trade_count": 0,
                    "winner_count": 0,
                    "loser_count": 0,
                    "win_rate_pct": 0.0,
                    "avg_trade_pnl_usd": 0.0,
                    "avg_holding_minutes": 0.0,
                    "median_holding_minutes": 0.0,
                }
            )
            continue

        trading = pair_series[row.symbol]
        trading = trading[trading["week_bucket"] == "trading"].copy()
        summary, trades_df = simulate_threshold_strategy(
            trading,
            entry_threshold_bps=float(row.optimal_entry_threshold_bps),
            target_spread_bps=float(row.target_spread_bps),
            starting_capital=starting_capital,
        )
        trade_logs[row.symbol] = trades_df
        trade_summaries.append(
            {
                "symbol": row.symbol,
                "strategy_type": "pair",
                "execution_style": EXECUTION_STYLE,
                "optimal_entry_threshold_bps": float(row.optimal_entry_threshold_bps),
                "target_spread_bps": float(row.target_spread_bps),
                "starting_balance_usd": starting_capital,
                "spot_allocation_usd": starting_capital * CAPITAL_SPLIT_SPOT,
                "futures_allocation_usd": starting_capital * CAPITAL_SPLIT_FUTURES,
                "ending_balance_usd": summary["ending_balance_usd"],
                "net_pnl_usd": summary["net_pnl_usd"],
                "return_pct": summary["return_pct"],
                "trade_count": summary["trade_count"],
                "winner_count": summary["winner_count"],
                "loser_count": summary["loser_count"],
                "win_rate_pct": summary["win_rate_pct"],
                "avg_trade_pnl_usd": summary["avg_trade_pnl_usd"],
                "avg_holding_minutes": summary["avg_holding_minutes"],
                "median_holding_minutes": summary["median_holding_minutes"],
            }
        )

    trade_summary = pd.DataFrame(trade_summaries).sort_values(
        ["ending_balance_usd", "trade_count"],
        ascending=[False, False],
    )
    return trade_summary, trade_logs


pair_trade_summary, pair_trade_logs = run_trading_week_backtest(
    pair_summary,
    pair_series,
)

print(
    "Executed out-of-sample trading-week backtest using thresholds optimized on the prior week, 50/50 spot-futures capital allocation, and taker fees"
)
display(pair_trade_summary)

display(
    pair_trade_summary[[
        "symbol",
        "execution_style",
        "optimal_entry_threshold_bps",
        "target_spread_bps",
        "spot_allocation_usd",
        "futures_allocation_usd",
        "ending_balance_usd",
        "net_pnl_usd",
        "return_pct",
        "trade_count",
        "winner_count",
        "loser_count",
        "win_rate_pct",
        "avg_holding_minutes",
        "median_holding_minutes",
    ]].sort_values("ending_balance_usd", ascending=False)
)

best_pair = pair_trade_summary.head(3)["symbol"].tolist() if not pair_trade_summary.empty else []
if best_pair:
    display({symbol: pair_trade_logs[symbol].head(10) for symbol in best_pair})

print(
    "Assumptions: 1-minute bars are cached to parquet in notebooks/data; the first full week calibrates the target spread and per-market entry threshold; the second full week trades those fixed parameters without lookahead bias; each trade allocates 50 percent of current balance to spot and 50 percent to futures and applies taker fees on both entry and exit."
)

Executed out-of-sample trading-week backtest using thresholds optimized on the prior week, 50/50 spot-futures capital allocation, and taker fees


,symbol,strategy_type,execution_style,optimal_entry_threshold_bps,target_spread_bps,starting_balance_usd,spot_allocation_usd,futures_allocation_usd,ending_balance_usd,net_pnl_usd,return_pct,trade_count,winner_count,loser_count,win_rate_pct,avg_trade_pnl_usd,avg_holding_minutes,median_holding_minutes
0,ATAUSDT,pair,taker,99.009901,13.359356,1000.0,500.0,500.0,449205.919466,448205.919466,44820.591947,1296,1296,0,100.000000,345.837901,2.105710,1.0
1,ALICEUSDT,pair,taker,8.857396,-21.186034,1000.0,500.0,500.0,67072.897971,66072.897971,6607.289797,1263,1263,0,100.000000,52.314250,3.977039,2.0
2,DYDXUSDT,pair,taker,20.661157,-13.416279,1000.0,500.0,500.0,54171.891090,53171.891090,5317.189109,1090,1090,0,100.000000,48.781551,2.492661,2.0
3,1000SATSUSDT,pair,taker,16.835017,-18.473382,1000.0,500.0,500.0,36638.885164,35638.885164,3563.888516,1079,1079,0,100.000000,33.029551,4.490269,2.0
4,GTCUSDT,pair,taker,113.636364,-7.083616,1000.0,500.0,500.0,34640.466245,33640.466245,3364.046625,306,305,1,99.673203,109.936164,13.901961,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
400,FLOWUSDT,pair,taker,NaN,0.087943,1000.0,500.0,500.0,1000.000000,0.000000,0.000000,0,0,0,0.000000,0.000000,0.000000,0.0
401,APEUSDT,pair,taker,NaN,-13.860421,1000.0,500.0,500.0,1000.000000,0.000000,0.000000,0,0,0,0.000000,0.000000,0.000000,0.0
402,NXPCUSDT,pair,taker,NaN,-17.339664,1000.0,500.0,500.0,1000.000000,0.000000,0.000000,0,0,0,0.000000,0.000000,0.000000,0.0
403,ERAUSDT,pair,taker,NaN,-28.065883,1000.0,500.0,500.0,1000.000000,0.000000,0.000000,0,0,0,0.000000,0.000000,0.000000,0.0


,symbol,execution_style,optimal_entry_threshold_bps,target_spread_bps,spot_allocation_usd,futures_allocation_usd,ending_balance_usd,net_pnl_usd,return_pct,trade_count,winner_count,loser_count,win_rate_pct,avg_holding_minutes,median_holding_minutes
0,ATAUSDT,taker,99.009901,13.359356,500.0,500.0,449205.919466,448205.919466,44820.591947,1296,1296,0,100.000000,2.105710,1.0
1,ALICEUSDT,taker,8.857396,-21.186034,500.0,500.0,67072.897971,66072.897971,6607.289797,1263,1263,0,100.000000,3.977039,2.0
2,DYDXUSDT,taker,20.661157,-13.416279,500.0,500.0,54171.891090,53171.891090,5317.189109,1090,1090,0,100.000000,2.492661,2.0
3,1000SATSUSDT,taker,16.835017,-18.473382,500.0,500.0,36638.885164,35638.885164,3563.888516,1079,1079,0,100.000000,4.490269,2.0
4,GTCUSDT,taker,113.636364,-7.083616,500.0,500.0,34640.466245,33640.466245,3364.046625,306,305,1,99.673203,13.901961,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,GIGGLEUSDT,taker,NaN,-11.977388,500.0,500.0,1000.000000,0.000000,0.000000,0,0,0,0.000000,0.000000,0.0
324,FETUSDT,taker,NaN,-14.877971,500.0,500.0,1000.000000,0.000000,0.000000,0,0,0,0.000000,0.000000,0.0
323,WLDUSDT,taker,NaN,-7.350365,500.0,500.0,1000.000000,0.000000,0.000000,0,0,0,0.000000,0.000000,0.0
322,XRPUSDC,taker,NaN,-5.788070,500.0,500.0,1000.000000,0.000000,0.000000,0,0,0,0.000000,0.000000,0.0


{'ATAUSDT':     symbol                entry_time                 exit_time  entry_spread_bps  exit_spread_bps  target_spread_bps  entry_threshold_bps  \
 0  ATAUSDT 2026-03-30 00:07:00+00:00 2026-03-30 00:09:00+00:00        120.481928              0.0          13.359356            99.009901   
 1  ATAUSDT 2026-03-30 00:11:00+00:00 2026-03-30 00:12:00+00:00        119.047619              0.0          13.359356            99.009901   
 2  ATAUSDT 2026-03-30 00:25:00+00:00 2026-03-30 00:26:00+00:00        117.647059              0.0          13.359356            99.009901   
 3  ATAUSDT 2026-03-30 00:31:00+00:00 2026-03-30 00:33:00+00:00        116.279070              0.0          13.359356            99.009901   
 4  ATAUSDT 2026-03-30 00:42:00+00:00 2026-03-30 00:43:00+00:00        117.647059              0.0          13.359356            99.009901   
 5  ATAUSDT 2026-03-30 00:44:00+00:00 2026-03-30 00:45:00+00:00        117.647059              0.0          13.359356            99.00990

Assumptions: 1-minute bars are cached to parquet in notebooks/data; the first full week calibrates the target spread and per-market entry threshold; the second full week trades those fixed parameters without lookahead bias; each trade allocates 50 percent of current balance to spot and 50 percent to futures and applies taker fees on both entry and exit.


In [59]:
SELECTED_PAIR_SYMBOL = 'GTCUSDT'

if SELECTED_PAIR_SYMBOL is None:
    print("No pair available for trade inspection.")
else:
    selected_config = pair_summary.loc[pair_summary["symbol"] == SELECTED_PAIR_SYMBOL].iloc[0]
    selected_trading = pair_series[SELECTED_PAIR_SYMBOL]
    selected_trading = selected_trading[selected_trading["week_bucket"] == "trading"].copy()
    selected_summary, selected_trades = simulate_threshold_strategy(
        selected_trading,
        entry_threshold_bps=float(selected_config["optimal_entry_threshold_bps"]),
        target_spread_bps=float(selected_config["target_spread_bps"]),
        starting_capital=STARTING_CAPITAL,
    )

    equity_curve = selected_trading[["timestamp", "spread_bps", "spot_close", "perp_close"]].copy()
    equity_curve["balance_usd"] = STARTING_CAPITAL
    if not selected_trades.empty:
        balance_steps = selected_trades[["exit_time", "ending_balance"]].rename(
            columns={"exit_time": "timestamp", "ending_balance": "balance_usd"}
        )
        equity_curve = equity_curve.merge(balance_steps, on="timestamp", how="left", suffixes=("", "_step"))
        equity_curve["balance_usd"] = equity_curve["balance_usd_step"].combine_first(equity_curve["balance_usd"])
        equity_curve = equity_curve.drop(columns=["balance_usd_step"])
    equity_curve["balance_usd"] = equity_curve["balance_usd"].ffill()

    print(f"Selected pair: {SELECTED_PAIR_SYMBOL}")
    print(
        f"Trading-week summary: ending balance={selected_summary['ending_balance_usd']:.2f} USD, "
        f"net pnl={selected_summary['net_pnl_usd']:.2f} USD, trades={selected_summary['trade_count']}"
    )
    print(
        "Position handling: only one trade can be open at a time. "
        "The simulator does not enter a new trade while the current balance is already allocated to the open spot/perp position."
    )

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=(
            f"{SELECTED_PAIR_SYMBOL} spread and calibrated levels",
            f"{SELECTED_PAIR_SYMBOL} trading-week equity curve"
        ),
    )
    fig.add_trace(
        go.Scatter(
            x=equity_curve["timestamp"],
            y=equity_curve["spread_bps"],
            mode="lines",
            name="spread_bps",
        ),
        row=1,
        col=1,
    )
    fig.add_hline(
        y=float(selected_config["target_spread_bps"]),
        line_width=1,
        line_dash="dot",
        line_color="gray",
        row=1,
        col=1,
    )
    fig.add_hline(
        y=float(selected_config["optimal_entry_threshold_bps"]),
        line_width=1,
        line_dash="dash",
        line_color="crimson",
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=equity_curve["timestamp"],
            y=equity_curve["balance_usd"],
            mode="lines",
            name="balance_usd",
        ),
        row=2,
        col=1,
    )
    fig.update_layout(
        height=800,
        width=1200,
        template="plotly_white",
        title=f"Trade inspection for {SELECTED_PAIR_SYMBOL}",
    )
    fig.show()

    display(
        selected_trades[[
            "entry_time",
            "exit_time",
            "entry_spread_bps",
            "exit_spread_bps",
            "spot_capital_usd",
            "futures_capital_usd",
            "gross_pnl_usd",
            "fees_usd",
            "net_pnl_usd",
            "ending_balance",
            "holding_bars",
            "holding_minutes",
            "winner",
        ]] if not selected_trades.empty else selected_trades
    )

Selected pair: GTCUSDT
Trading-week summary: ending balance=34640.47 USD, net pnl=33640.47 USD, trades=306
Position handling: only one trade can be open at a time. The simulator does not enter a new trade while the current balance is already allocated to the open spot/perp position.


,entry_time,exit_time,entry_spread_bps,exit_spread_bps,spot_capital_usd,futures_capital_usd,gross_pnl_usd,fees_usd,net_pnl_usd,ending_balance,holding_bars,holding_minutes,winner
0,2026-03-30 00:01:00+00:00,2026-03-30 00:06:00+00:00,133.333333,-131.578947,500.000000,500.000000,13.245614,1.503377,11.742237,1011.742237,5,5.0,True
1,2026-03-30 00:17:00+00:00,2026-03-30 00:24:00+00:00,133.333333,-131.578947,505.871118,505.871118,13.401147,1.521030,11.880117,1023.622354,7,7.0,True
2,2026-03-30 00:49:00+00:00,2026-03-30 00:51:00+00:00,133.333333,-131.578947,511.811177,511.811177,13.558507,1.538891,12.019616,1035.641970,2,2.0,True
3,2026-03-30 01:18:00+00:00,2026-03-30 01:34:00+00:00,133.333333,-131.578947,517.820985,517.820985,13.717714,1.556961,12.160753,1047.802723,16,16.0,True
4,2026-03-30 01:58:00+00:00,2026-03-30 02:21:00+00:00,133.333333,-131.578947,523.901362,523.901362,13.878790,1.575243,12.303548,1060.106271,23,23.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,2026-04-05 20:56:00+00:00,2026-04-05 21:04:00+00:00,133.333333,-131.578947,16554.883965,16554.883965,438.559207,49.776470,388.782737,33498.550667,8,8.0,True
302,2026-04-05 21:08:00+00:00,2026-04-05 21:51:00+00:00,133.333333,-131.578947,16749.275334,16749.275334,443.708873,50.360957,393.347916,33891.898583,43,43.0,True
303,2026-04-05 21:57:00+00:00,2026-04-05 22:06:00+00:00,133.333333,-131.578947,16945.949291,16945.949291,448.919008,50.952307,397.966700,34289.865283,9,9.0,True
304,2026-04-05 22:16:00+00:00,2026-04-05 22:49:00+00:00,133.333333,-131.578947,17144.932642,17144.932642,454.190321,51.550601,402.639719,34692.505003,33,33.0,True


In [61]:
from IPython.display import HTML

pair_trade_summary_executed = pair_trade_summary.loc[
    pair_trade_summary["trade_count"] > 0
].sort_values(["ending_balance_usd", "trade_count"], ascending=[False, False])

print(f"Pairs with at least one executed trade: {len(pair_trade_summary_executed):,}")
display(HTML(pair_trade_summary_executed.to_html(index=False)))

Pairs with at least one executed trade: 292


symbol,strategy_type,execution_style,optimal_entry_threshold_bps,target_spread_bps,starting_balance_usd,spot_allocation_usd,futures_allocation_usd,ending_balance_usd,net_pnl_usd,return_pct,trade_count,winner_count,loser_count,win_rate_pct,avg_trade_pnl_usd,avg_holding_minutes,median_holding_minutes
ATAUSDT,pair,taker,99.009901,13.359356,1000.0,500.0,500.0,449205.919466,448205.919466,44820.591947,1296,1296,0,100.000000,345.837901,2.105710,1.0
ALICEUSDT,pair,taker,8.857396,-21.186034,1000.0,500.0,500.0,67072.897971,66072.897971,6607.289797,1263,1263,0,100.000000,52.314250,3.977039,2.0
DYDXUSDT,pair,taker,20.661157,-13.416279,1000.0,500.0,500.0,54171.891090,53171.891090,5317.189109,1090,1090,0,100.000000,48.781551,2.492661,2.0
1000SATSUSDT,pair,taker,16.835017,-18.473382,1000.0,500.0,500.0,36638.885164,35638.885164,3563.888516,1079,1079,0,100.000000,33.029551,4.490269,2.0
GTCUSDT,pair,taker,113.636364,-7.083616,1000.0,500.0,500.0,34640.466245,33640.466245,3364.046625,306,305,1,99.673203,109.936164,13.901961,9.0
TRUUSDT,pair,taker,58.333333,28.212774,1000.0,500.0,500.0,12999.032431,11999.032431,1199.903243,309,309,0,100.000000,38.831820,16.291262,9.0
PHBUSDT,pair,taker,35.087719,2.365392,1000.0,500.0,500.0,8424.931724,7424.931724,742.493172,564,564,0,100.000000,13.164773,7.244681,5.0
ARKMUSDT,pair,taker,28.037383,-7.296318,1000.0,500.0,500.0,6786.116112,5786.116112,578.611611,523,522,1,99.808795,11.063320,8.118547,5.0
DENTUSDT,pair,taker,45.662100,-26.326679,1000.0,500.0,500.0,6592.039722,5592.039722,559.203972,431,431,0,100.000000,12.974570,11.494200,6.0
IOUSDT,pair,taker,25.641026,-9.767826,1000.0,500.0,500.0,4655.525112,3655.525112,365.552511,462,462,0,100.000000,7.912392,10.166667,7.0
